# Phase 2: Data Enrichment via Web Scraping

| | |
|---|---|
| **What** | Change the ICD-9 codes in `diag_1` (like `428`) into disease names by scraping a website. |
| **Why** | A code like `428` means nothing to a business reader. "Heart failure" is easy to understand. |

## Part 1: Loading the Data

| | |
|---|---|
| **What** | Import the libraries and load `01_cleaned.csv` from Phase 1. `diag_1` is read as text. |
| **Why** | Some codes have letters or decimals (`V57`, `250.8`). As numbers, they could get changed. |

In [1]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

# Load Phase 1 output, keep diag_1 as text and 'None' labels as text
df = pd.read_csv("../data/processed/01_cleaned.csv", dtype={"diag_1": str},
                 keep_default_na=False, na_values=[''], low_memory=False)

df.shape

(100114, 49)

## Part 2: Fixing the Leading Zeros

| | |
|---|---|
| **What** | Pad number codes to 3 digits (`38` → `038`, `8` → `008`). `V`, `E` and decimal codes stay the same. |
| **Why** | ICD-9 number codes have 3 digits, but the zeros were lost earlier. Without them the website cannot find codes like `038` (Septicemia). |
| **Checked** | The data has no short decimal codes like `8.45`, so this fix is safe. |

In [5]:
# Pad numeric codes to 3 digits: '38' -> '038' , '8' -> '008'
# V/E codes (V57) and decimal codes (250.8) stay the same

df["diag_1"] = df["diag_1"].apply(
    lambda c: c.zfill(3) if isinstance(c, str) and c[:1].isdigit() else c
)

df["diag_1"].head()

0    250.83
1       276
2       648
3       008
4       197
Name: diag_1, dtype: str